# Demo Modul 1: Fondasi Jaringan Saraf, FNN, Aktivasi, dan Loss

**Durasi sesi:** 120 menit  
**Kasus:** klasifikasi biner dua bulan sabit (`make_moons`)  
**Fokus:** neuron, shape, aktivasi, forward pass, logits, probabilitas, dan loss.

> Training loop dipakai sebagai utilitas pada modul ini. Mekanisme gradien dan backpropagation dibahas pada Modul 2.

## Capaian demo

Setelah demo, praktikan dapat:

1. menghitung satu neuron untuk satu batch;
2. membandingkan sigmoid, tanh, ReLU, dan Leaky ReLU;
3. menghitung forward pass FNN sampai BCE;
4. mencocokkan NumPy dengan PyTorch; dan
5. melatih serta mengevaluasi FNN sederhana tanpa data leakage.

In [ ]:
import platform
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
from sklearn.datasets import make_moons
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
pd.set_option('display.precision', 4)
print({
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'sklearn': sklearn.__version__,
    'torch': torch.__version__,
    'device': str(DEVICE),
    'seed': SEED,
})

## 1. Neuron untuk satu batch

Dengan observasi sebagai baris, $X$ berbentuk $(B,d)$, $w$ berbentuk $(d,)$, dan keluaran berbentuk $(B,)$.

In [ ]:
def sigmoid_np(z: np.ndarray) -> np.ndarray:
    z = np.asarray(z, dtype=np.float64)
    positive = z >= 0
    result = np.empty_like(z)
    result[positive] = 1.0 / (1.0 + np.exp(-z[positive]))
    exp_z = np.exp(z[~positive])
    result[~positive] = exp_z / (1.0 + exp_z)
    return result

def neuron_batch(X: np.ndarray, w: np.ndarray, b: float, activation):
    X = np.asarray(X, dtype=np.float64)
    w = np.asarray(w, dtype=np.float64)
    if X.ndim != 2 or w.ndim != 1 or X.shape[1] != w.shape[0]:
        raise ValueError(f'Shape tidak cocok: X={X.shape}, w={w.shape}')
    z = X @ w + b
    return z, activation(z)

X_small = np.array([[2.0, -1.0], [0.0, 3.0], [-2.0, 1.0]])
w_small = np.array([0.5, -0.5])
z_small, a_small = neuron_batch(X_small, w_small, 0.25, sigmoid_np)
pd.DataFrame({'z': z_small, 'sigmoid(z)': a_small})

**Pemeriksaan:** bias ditambahkan ke setiap observasi melalui broadcasting; fungsi aktivasi tidak mengubah shape.

## 2. Membandingkan fungsi aktivasi

Aktivasi hidden layer memberi nonlinearitas. Aktivasi keluaran dipilih berdasarkan makna target.

In [ ]:
def relu_np(z):
    return np.maximum(0.0, z)

def leaky_relu_np(z, alpha=0.01):
    return np.where(z >= 0, z, alpha * z)

grid = np.linspace(-6, 6, 600)
activation_values = {
    'Sigmoid': sigmoid_np(grid),
    'Tanh': np.tanh(grid),
    'ReLU': relu_np(grid),
    'Leaky ReLU': leaky_relu_np(grid),
}

fig, axes = plt.subplots(2, 2, figsize=(10, 6), sharex=True)
for axis, (name, values) in zip(axes.ravel(), activation_values.items()):
    axis.plot(grid, values, color='#4B6DB4', linewidth=2)
    axis.axhline(0, color='black', linewidth=0.7)
    axis.axvline(0, color='black', linewidth=0.7)
    axis.set_title(name)
    axis.set_xlabel('z')
    axis.set_ylabel('phi(z)')
    axis.grid(alpha=0.2)
fig.suptitle('Perbandingan Fungsi Aktivasi')
fig.tight_layout()
plt.show()

## 3. Forward pass manual: $2 \rightarrow 2 \rightarrow 1$

Target adalah kelas positif ($y=1$). Hidden layer memakai ReLU dan output model adalah satu logit.

In [ ]:
x_manual = np.array([[2.0, -1.0]], dtype=np.float32)
W1 = np.array([[0.5, -0.5], [1.0, 1.0]], dtype=np.float32)
b1 = np.array([0.0, 0.0], dtype=np.float32)
W2 = np.array([[2.0, -1.0]], dtype=np.float32)
b2 = np.array([0.5], dtype=np.float32)
y_manual = np.array([[1.0]], dtype=np.float32)

z1_np = x_manual @ W1.T + b1
h_np = relu_np(z1_np)
logit_np = h_np @ W2.T + b2
prob_np = sigmoid_np(logit_np)
bce_np = -(y_manual * np.log(prob_np) + (1-y_manual) * np.log(1-prob_np)).mean()
parameter_count_manual = W1.size + b1.size + W2.size + b2.size

manual_result = pd.DataFrame({
    'besaran': ['z1', 'h=ReLU(z1)', 'logit', 'probabilitas', 'BCE', 'jumlah parameter'],
    'nilai': [z1_np.tolist(), h_np.tolist(), float(logit_np.item()),
              float(prob_np.item()), float(bce_np), parameter_count_manual],
    'shape': [str(z1_np.shape), str(h_np.shape), str(logit_np.shape),
              str(prob_np.shape), 'skalar', 'skalar'],
})
manual_result

## 4. Pencocokan NumPy dan PyTorch

Bobot yang sama harus menghasilkan logit dan loss yang sama. `BCEWithLogitsLoss` menerima logit secara langsung.

In [ ]:
manual_model = nn.Sequential(
    nn.Linear(2, 2),
    nn.ReLU(),
    nn.Linear(2, 1),
)
with torch.no_grad():
    manual_model[0].weight.copy_(torch.from_numpy(W1))
    manual_model[0].bias.copy_(torch.from_numpy(b1))
    manual_model[2].weight.copy_(torch.from_numpy(W2))
    manual_model[2].bias.copy_(torch.from_numpy(b2))

x_t = torch.from_numpy(x_manual)
y_t = torch.from_numpy(y_manual)
torch_logit = manual_model(x_t)
torch_prob = torch.sigmoid(torch_logit)
torch_loss = nn.BCEWithLogitsLoss()(torch_logit, y_t)

max_logit_diff = float(np.max(np.abs(torch_logit.detach().numpy() - logit_np)))
max_loss_diff = abs(torch_loss.item() - float(bce_np))
assert max_logit_diff < 1e-6
assert max_loss_diff < 1e-6
print({
    'torch_logit': torch_logit.item(),
    'torch_probability': torch_prob.item(),
    'torch_loss': torch_loss.item(),
    'max_logit_difference': max_logit_diff,
    'max_loss_difference': max_loss_diff,
    'parameters': sum(p.numel() for p in manual_model.parameters()),
})

### Checkpoint menit ke-85

Tunjukkan plot empat aktivasi, tabel shape dan parameter, serta selisih NumPy-PyTorch kurang dari $10^{-6}$.

## 5. Data `make_moons` tanpa leakage

Split dilakukan sebelum standardisasi. `StandardScaler` hanya di-fit pada train set.

In [ ]:
X, y = make_moons(n_samples=600, noise=0.22, random_state=SEED)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train).astype(np.float32)
X_val_s = scaler.transform(X_val).astype(np.float32)
X_test_s = scaler.transform(X_test).astype(np.float32)
y_train_f = y_train.astype(np.float32).reshape(-1, 1)
y_val_f = y_val.astype(np.float32).reshape(-1, 1)
y_test_f = y_test.astype(np.float32).reshape(-1, 1)

split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'n': [len(y_train), len(y_val), len(y_test)],
    'positive_rate': [y_train.mean(), y_val.mean(), y_test.mean()],
})
display(split_summary)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(X_train_s[:, 0], X_train_s[:, 1], c=y_train, cmap='coolwarm', s=24, alpha=0.75)
ax.set(title='Train Set make_moons (Terstandardisasi)', xlabel='fitur 1', ylabel='fitur 2')
ax.grid(alpha=0.2)
plt.show()

## 6. Model dan utilitas training

Untuk perbandingan adil, fungsi di bawah selalu membuat model dan DataLoader baru dengan seed yang sama.

In [ ]:
def activation_layer(name: str) -> nn.Module:
    choices = {
        'relu': nn.ReLU,
        'tanh': nn.Tanh,
        'sigmoid': nn.Sigmoid,
        'leaky_relu': lambda: nn.LeakyReLU(negative_slope=0.01),
    }
    if name not in choices:
        raise ValueError(f'Aktivasi tidak dikenal: {name}')
    return choices[name]()

def build_model(hidden_dim=8, activation='relu', seed=SEED):
    seed_everything(seed)
    return nn.Sequential(
        nn.Linear(2, hidden_dim),
        activation_layer(activation),
        nn.Linear(hidden_dim, 1),
    ).to(DEVICE)

def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())

baseline_model = build_model(hidden_dim=8, activation='relu')
print(baseline_model)
print('parameter aktual:', count_parameters(baseline_model))
print('parameter rumus:', 4 * 8 + 1)
assert count_parameters(baseline_model) == 4 * 8 + 1

In [ ]:
def as_tensor(array):
    return torch.as_tensor(array, dtype=torch.float32)

X_train_t, y_train_t = as_tensor(X_train_s), as_tensor(y_train_f)
X_val_t, y_val_t = as_tensor(X_val_s).to(DEVICE), as_tensor(y_val_f).to(DEVICE)
X_test_t, y_test_t = as_tensor(X_test_s).to(DEVICE), as_tensor(y_test_f).to(DEVICE)

def evaluate(model, X_tensor, y_tensor):
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        loss = nn.functional.binary_cross_entropy_with_logits(logits, y_tensor).item()
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).to(torch.int64)
        accuracy = (predictions == y_tensor.to(torch.int64)).float().mean().item()
    return {
        'loss': loss,
        'accuracy': accuracy,
        'probabilities': probabilities.cpu().numpy().ravel(),
        'predictions': predictions.cpu().numpy().ravel(),
    }

def train_model(model, epochs=200, learning_rate=0.05, batch_size=32, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
    )
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    criterion = nn.BCEWithLogitsLoss()
    history = []
    start = time.perf_counter()

    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum = 0.0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(X_batch)
        val_metrics = evaluate(model, X_val_t, y_val_t)
        history.append({
            'epoch': epoch,
            'train_loss': loss_sum / len(X_train_t),
            'val_loss': val_metrics['loss'],
            'val_accuracy': val_metrics['accuracy'],
        })

    return pd.DataFrame(history), time.perf_counter() - start

## 7. Baseline $2 \rightarrow 8 \rightarrow 1$ dengan ReLU

In [ ]:
baseline_model = build_model(hidden_dim=8, activation='relu', seed=SEED)
baseline_history, baseline_runtime = train_model(baseline_model, seed=SEED)
baseline_val = evaluate(baseline_model, X_val_t, y_val_t)
print({
    'parameters': count_parameters(baseline_model),
    'runtime_seconds': round(baseline_runtime, 3),
    'val_loss': round(baseline_val['loss'], 4),
    'val_accuracy': round(baseline_val['accuracy'], 4),
})

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(baseline_history['epoch'], baseline_history['train_loss'], label='train')
ax.plot(baseline_history['epoch'], baseline_history['val_loss'], label='validation')
ax.set(title='Kurva Loss Baseline', xlabel='epoch', ylabel='BCE loss')
ax.legend()
ax.grid(alpha=0.2)
plt.show()

In [ ]:
def plot_decision_boundary(model, X_values, y_values, title):
    x0_min, x0_max = X_values[:, 0].min() - 0.5, X_values[:, 0].max() + 0.5
    x1_min, x1_max = X_values[:, 1].min() - 0.5, X_values[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x0_min, x0_max, 220),
        np.linspace(x1_min, x1_max, 220),
    )
    mesh = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32, device=DEVICE)
    model.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(model(mesh)).cpu().numpy().reshape(xx.shape)
    fig, ax = plt.subplots(figsize=(6.5, 5))
    contour = ax.contourf(xx, yy, probabilities, levels=np.linspace(0, 1, 11), cmap='coolwarm', alpha=0.45)
    ax.contour(xx, yy, probabilities, levels=[0.5], colors='black', linewidths=1.5)
    ax.scatter(X_values[:, 0], X_values[:, 1], c=y_values, cmap='coolwarm', edgecolor='white', s=28)
    ax.set(title=title, xlabel='fitur 1', ylabel='fitur 2')
    fig.colorbar(contour, ax=ax, label='P(y=1)')
    plt.show()

plot_decision_boundary(baseline_model, X_val_s, y_val, 'Decision Boundary pada Validation Set')
cm = confusion_matrix(y_val, baseline_val['predictions'])
ConfusionMatrixDisplay(cm).plot(cmap='Blues', colorbar=False)
plt.title('Confusion Matrix Baseline - Validation')
plt.show()

## 8. Latihan individual di kelas

- Digit terakhir NIM 0-4: ganti ReLU dengan tanh.
- Digit terakhir NIM 5-9: ganti ReLU dengan sigmoid.

Pertahankan hidden size, split, optimizer, learning rate, batch size, epoch, dan seed. Tulis prediksi sebelum menjalankan variasi.

In [ ]:
# Contoh demonstrasi variasi terkontrol; mahasiswa memilih sesuai digit NIM.
variant_name = 'tanh'
variant_model = build_model(hidden_dim=8, activation=variant_name, seed=SEED)
variant_history, variant_runtime = train_model(variant_model, seed=SEED)
variant_val = evaluate(variant_model, X_val_t, y_val_t)
pd.DataFrame([
    {'activation': 'relu', 'hidden_dim': 8, 'parameters': count_parameters(baseline_model),
     'val_loss': baseline_val['loss'], 'val_accuracy': baseline_val['accuracy'],
     'runtime_seconds': baseline_runtime},
    {'activation': variant_name, 'hidden_dim': 8, 'parameters': count_parameters(variant_model),
     'val_loss': variant_val['loss'], 'val_accuracy': variant_val['accuracy'],
     'runtime_seconds': variant_runtime},
]).sort_values('val_loss')

## Exit ticket

1. Mengapa output terakhir tidak diberi sigmoid di dalam model?
2. Berapa parameter FNN $2 \rightarrow 16 \rightarrow 1$?
3. Bukti apa yang harus ditunjukkan sebelum menyatakan satu aktivasi lebih baik?

**Tugas setelah sesi:** jalankan enam konfigurasi pada starter mahasiswa, pilih model dari validation set, lalu gunakan test set satu kali.